In [ ]:
# ============================================================
# MIL-AEGIS
# External Attack Surface & Web Security Assessment Prototype
#
# AUTHORIZED USE ONLY
#
# Set TARGET exactly once below.
# ============================================================

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

TARGET = "https://www.lifehacks-investments.com"

# Optional: restrict scanning to the target hostname.
# This prevents discovered third-party URLs from becoming
# unintended scan targets.
ALLOWED_HOST = "lifehacks-investments.com"

# Maximum number of pages to crawl.
MAX_CRAWL_PAGES = 30

# Maximum links examined from a single page.
MAX_LINKS_PER_PAGE = 50

# Network timeout.
REQUEST_TIMEOUT = 8

# ------------------------------------------------------------
# HARD SAFETY LIMITS FOR RESILIENCE TESTING
# ------------------------------------------------------------
#
# These are deliberately conservative.
#
# The module is NOT a stress/DDoS tester.
# It performs a bounded resilience/rate-limit assessment.
#
MAX_RESILIENCE_DURATION = 30          # seconds
MAX_RESILIENCE_REQUESTS = 100
MAX_RESILIENCE_BURST = 10
MAX_RESILIENCE_CONCURRENCY = 2
MAX_RESILIENCE_TIMEOUT = 10
MAX_CONSECUTIVE_FAILURES = 3
MAX_5XX_RATIO = 0.10
MAX_LATENCY_MULTIPLIER = 3.0
RESILIENCE_COOLDOWN = 10
NORMAL_REQUEST_INTERVAL = 2.0

USER_AGENT = "MIL-AEGIS-Authorized-Assessment/1.0"

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys
import subprocess

PACKAGES = [
    "requests",
    "beautifulsoup4",
    "dnspython",
    "cryptography",
    "pandas",
    "streamlit",
]

for package in PACKAGES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        check=False
    )

# ============================================================
# 2. IMPORTS
# ============================================================

import re
import ssl
import json
import time
import socket
import hashlib
import statistics
import threading

from urllib.parse import (
    urljoin,
    urlparse,
    parse_qs,
    urlencode,
    urlunparse,
)

from collections import deque, defaultdict
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd

from bs4 import BeautifulSoup

# ============================================================
# 3. TARGET VALIDATION
# ============================================================

TARGET = TARGET.rstrip("/")

parsed_target = urlparse(TARGET)

if parsed_target.scheme not in ("http", "https"):
    raise ValueError("TARGET must use http:// or https://")

TARGET_HOST = parsed_target.hostname.lower()

if TARGET_HOST != ALLOWED_HOST.lower():
    raise RuntimeError(
        f"Target guard blocked execution.\n"
        f"TARGET host: {TARGET_HOST}\n"
        f"Allowed host: {ALLOWED_HOST}"
    )

print("=" * 70)
print("MIL-AEGIS")
print("=" * 70)
print(f"Authorized target : {TARGET}")
print(f"Allowed hostname  : {ALLOWED_HOST}")
print("=" * 70)


# ============================================================
# 4. GLOBAL RESULT STORE
# ============================================================

REPORT = {
    "target": TARGET,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "assets": [],
    "ports": [],
    "services": [],
    "pages": [],
    "forms": [],
    "cookies": [],
    "headers": [],
    "apis": [],
    "technologies": [],
    "findings": [],
    "cves": [],
    "kev": [],
    "resilience": {},
}


# ============================================================
# 5. COMMON HELPERS
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": USER_AGENT
})


def same_target_host(url):
    """Return True only if URL belongs to our authorized host."""
    try:
        host = urlparse(url).hostname
        if not host:
            return False

        host = host.lower()

        return (
            host == ALLOWED_HOST.lower()
            or host.endswith("." + ALLOWED_HOST.lower())
        )

    except Exception:
        return False


def normalize_url(url):
    """Normalize URL for crawl deduplication."""
    p = urlparse(url)

    return urlunparse((
        p.scheme.lower(),
        p.netloc.lower(),
        p.path or "/",
        "",
        p.query,
        "",
    ))


def add_finding(
    title,
    severity,
    category,
    description,
    url=None,
    confidence="Medium",
    evidence=None,
    cwe=None,
):
    finding = {
        "title": title,
        "severity": severity,
        "category": category,
        "description": description,
        "url": url,
        "confidence": confidence,
        "evidence": evidence,
        "cwe": cwe,
    }

    REPORT["findings"].append(finding)


def severity_rank(severity):
    return {
        "Critical": 4,
        "High": 3,
        "Medium": 2,
        "Low": 1,
        "Info": 0,
    }.get(severity, 0)


# ============================================================
# 6. DNS DISCOVERY
# ============================================================

def dns_discovery():
    print("\n[+] DNS discovery")

    hostname = TARGET_HOST

    addresses = set()

    try:
        results = socket.getaddrinfo(
            hostname,
            443,
            type=socket.SOCK_STREAM
        )

        for result in results:
            addresses.add(result[4][0])

    except Exception as exc:
        print("DNS error:", exc)

    asset = {
        "hostname": hostname,
        "addresses": sorted(addresses),
    }

    REPORT["assets"].append(asset)

    print("    Addresses:", sorted(addresses))

    return sorted(addresses)


# ============================================================
# 7. TCP PORT DISCOVERY
# ============================================================

def port_scan(host):
    """
    Conservative TCP connect scan.

    This intentionally uses connect() rather than raw packet
    crafting or stealth techniques.
    """

    print("\n[+] TCP service discovery")

    # Common web/application ports.
    ports = [
        21,
        22,
        25,
        53,
        80,
        110,
        143,
        443,
        465,
        587,
        993,
        995,
        3000,
        5000,
        8000,
        8080,
        8443,
        8888,
        9200,
        27017,
        3306,
        5432,
        6379,
    ]

    open_ports = []

    for port in ports:

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        sock.settimeout(1.5)

        try:
            result = sock.connect_ex((host, port))

            if result == 0:
                open_ports.append(port)

                REPORT["ports"].append({
                    "host": host,
                    "port": port,
                    "state": "open",
                })

                print(f"    OPEN {port}")

        except Exception:
            pass

        finally:
            sock.close()

    return open_ports


# ============================================================
# 8. SERVICE/BANNER FINGERPRINTING
# ============================================================

def service_fingerprint(host, port):
    """
    Safe service identification.

    For HTTP/HTTPS, use HTTP/TLS inspection.
    For other ports, collect only a small initial response.
    """

    result = {
        "host": host,
        "port": port,
        "service": "unknown",
        "banner": None,
    }

    if port in (80, 8080, 8000, 8888):
        result["service"] = "HTTP"

    elif port in (443, 8443):
        result["service"] = "HTTPS"

    elif port == 22:
        result["service"] = "SSH"

    elif port == 21:
        result["service"] = "FTP"

    elif port == 25:
        result["service"] = "SMTP"

    elif port == 3306:
        result["service"] = "MySQL"

    elif port == 5432:
        result["service"] = "PostgreSQL"

    elif port == 6379:
        result["service"] = "Redis"

    elif port == 27017:
        result["service"] = "MongoDB"

    elif port == 9200:
        result["service"] = "Elasticsearch"

    REPORT["services"].append(result)

    return result


# ============================================================
# 9. HTTP/TLS INSPECTION
# ============================================================

def inspect_http(url):
    print("\n[+] HTTP inspection:", url)

    try:
        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )

    except Exception as exc:
        print("    Request failed:", exc)
        return None

    headers = {
        k.lower(): v
        for k, v in response.headers.items()
    }

    page = {
        "url": url,
        "final_url": response.url,
        "status": response.status_code,
        "headers": dict(response.headers),
        "content_type": headers.get("content-type"),
        "length": len(response.content),
    }

    REPORT["pages"].append(page)

    return response


# ============================================================
# 10. SECURITY HEADER ANALYSIS
# ============================================================

SECURITY_HEADERS = {
    "strict-transport-security": "HSTS",
    "content-security-policy": "CSP",
    "x-content-type-options": "X-Content-Type-Options",
    "x-frame-options": "X-Frame-Options",
    "referrer-policy": "Referrer-Policy",
    "permissions-policy": "Permissions-Policy",
}


def analyze_security_headers(response):

    if not response:
        return

    headers = {
        k.lower(): v
        for k, v in response.headers.items()
    }

    for header, friendly_name in SECURITY_HEADERS.items():

        present = header in headers

        REPORT["headers"].append({
            "url": response.url,
            "header": friendly_name,
            "present": present,
            "value": headers.get(header),
        })

        if not present:

            # Do not call every missing header a vulnerability.
            add_finding(
                title=f"Missing {friendly_name}",
                severity="Low",
                category="Security Headers",
                description=(
                    f"{friendly_name} was not observed in the HTTP response."
                ),
                url=response.url,
                confidence="High",
            )


# ============================================================
# 11. COOKIE ANALYSIS
# ============================================================

def analyze_cookies(response):

    if not response:
        return

    for cookie in response.cookies:

        record = {
            "name": cookie.name,
            "domain": cookie.domain,
            "path": cookie.path,
            "secure": cookie.secure,
            "value_length": len(cookie.value),
        }

        REPORT["cookies"].append(record)

        # requests does not expose every Set-Cookie attribute
        # conveniently, so inspect raw headers as well.
        raw_set_cookie = response.headers.get("Set-Cookie", "")

        cookie_name = re.escape(cookie.name)

        match = re.search(
            rf"(?:^|,)\s*{cookie_name}=[^,]*",
            raw_set_cookie,
            re.IGNORECASE
        )

        cookie_segment = match.group(0) if match else raw_set_cookie

        http_only = bool(
            re.search(
                r"\bHttpOnly\b",
                cookie_segment,
                re.IGNORECASE
            )
        )

        same_site = re.search(
            r"\bSameSite=([^;,]+)",
            cookie_segment,
            re.IGNORECASE
        )

        same_site_value = (
            same_site.group(1)
            if same_site
            else None
        )

        if not cookie.secure and response.url.startswith("https://"):

            add_finding(
                title="Cookie without Secure attribute",
                severity="Medium",
                category="Cookies",
                description=(
                    "A cookie was observed without the Secure attribute "
                    "on an HTTPS response."
                ),
                url=response.url,
                confidence="High",
                evidence=cookie.name,
            )

        if not http_only:

            # Don't automatically treat analytics cookies as a vulnerability.
            if re.search(
                r"(session|auth|token|login)",
                cookie.name,
                re.IGNORECASE
            ):
                add_finding(
                    title="Potentially sensitive cookie lacks HttpOnly",
                    severity="Medium",
                    category="Cookies",
                    description=(
                        "A cookie with a session/authentication-like name "
                        "was observed without HttpOnly."
                    ),
                    url=response.url,
                    confidence="Medium",
                    evidence=cookie.name,
                )


# ============================================================
# 12. CORS ANALYSIS
# ============================================================

def analyze_cors(response):

    if not response:
        return

    headers = {
        k.lower(): v
        for k, v in response.headers.items()
    }

    origin = headers.get(
        "access-control-allow-origin"
    )

    credentials = headers.get(
        "access-control-allow-credentials"
    )

    if origin:

        record = {
            "url": response.url,
            "allow_origin": origin,
            "allow_credentials": credentials,
        }

        REPORT["apis"].append({
            "type": "CORS",
            **record
        })

        if origin == "*" and credentials == "true":

            add_finding(
                title="Potentially dangerous CORS configuration",
                severity="High",
                category="CORS",
                description=(
                    "Wildcard Access-Control-Allow-Origin was observed "
                    "together with credentials."
                ),
                url=response.url,
                confidence="High",
            )


# ============================================================
# 13. WEB CRAWLER
# ============================================================

def crawl():

    print("\n[+] Web crawling")

    queue = deque([normalize_url(TARGET)])
    visited = set()

    while queue and len(visited) < MAX_CRAWL_PAGES:

        url = queue.popleft()

        if url in visited:
            continue

        if not same_target_host(url):
            continue

        visited.add(url)

        response = inspect_http(url)

        if not response:
            continue

        analyze_security_headers(response)
        analyze_cookies(response)
        analyze_cors(response)

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # ----------------------------------------------------
        # Forms
        # ----------------------------------------------------

        for form in soup.find_all("form"):

            action = form.get("action") or url

            action = urljoin(url, action)

            method = (
                form.get("method", "GET")
                .upper()
            )

            inputs = []

            for inp in form.find_all("input"):

                inputs.append({
                    "name": inp.get("name"),
                    "type": inp.get("type"),
                })

            form_record = {
                "page": url,
                "action": action,
                "method": method,
                "inputs": inputs,
            }

            REPORT["forms"].append(form_record)

            # Authentication discovery.
            input_types = [
                (i.get("type") or "").lower()
                for i in inputs
            ]

            names = [
                (i.get("name") or "").lower()
                for i in inputs
            ]

            if (
                "password" in input_types
                or any(
                    "password" in n
                    for n in names
                )
            ):

                REPORT["apis"].append({
                    "type": "Authentication surface",
                    "url": action,
                    "method": method,
                })

        # ----------------------------------------------------
        # Links
        # ----------------------------------------------------

        links = soup.find_all("a", href=True)

        for link in links[:MAX_LINKS_PER_PAGE]:

            href = urljoin(
                response.url,
                link["href"]
            )

            if same_target_host(href):

                normalized = normalize_url(href)

                if normalized not in visited:
                    queue.append(normalized)

        # ----------------------------------------------------
        # JavaScript resources
        # ----------------------------------------------------

        for script in soup.find_all("script", src=True):

            src = urljoin(
                response.url,
                script["src"]
            )

            if same_target_host(src):

                analyze_javascript(src)

    print(
        f"    Crawled {len(visited)} pages"
    )


# ============================================================
# 14. JAVASCRIPT ANALYSIS
# ============================================================

SECRET_PATTERNS = {

    "AWS-like access key":
        r"\bAKIA[0-9A-Z]{16}\b",

    "Private key material":
        r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",

    "JWT-like token":
        r"\beyJ[a-zA-Z0-9_-]{10,}\.[a-zA-Z0-9_-]{10,}\.[a-zA-Z0-9_-]{10,}\b",

    "Generic API key":
        r"(?i)(api[_-]?key|apikey)\s*[:=]\s*['\"][A-Za-z0-9_\-]{16,}['\"]",

    "Generic secret":
        r"(?i)(secret|password|passwd|token)\s*[:=]\s*['\"][^'\"]{8,}['\"]",
}


def analyze_javascript(url):

    try:
        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT
        )

    except Exception:
        return

    if response.status_code != 200:
        return

    text = response.text

    # --------------------------------------------------------
    # API endpoint discovery
    # --------------------------------------------------------

    endpoint_patterns = [
        r"""["'](/api/[A-Za-z0-9_./?=&%-]+)["']""",
        r"""["'](https?://[^"' ]+/api/[^"' ]*)["']""",
        r"""["'](/graphql[^"' ]*)["']""",
    ]

    for pattern in endpoint_patterns:

        for match in re.findall(pattern, text):

            endpoint = urljoin(url, match)

            if same_target_host(endpoint):

                REPORT["apis"].append({
                    "type": "JavaScript API endpoint",
                    "url": endpoint,
                    "source": url,
                })

    # --------------------------------------------------------
    # Secret-like patterns
    # --------------------------------------------------------

    for name, pattern in SECRET_PATTERNS.items():

        matches = re.findall(
            pattern,
            text
        )

        if matches:

            add_finding(
                title=f"Potential exposed {name}",
                severity="High",
                category="Code / Secrets",
                description=(
                    "A secret-like pattern was found in a publicly "
                    "accessible JavaScript resource. Manual validation "
                    "is required."
                ),
                url=url,
                confidence="Low",
                evidence=f"{len(matches)} potential match(es)",
            )

    # --------------------------------------------------------
    # Development/debug indicators
    # --------------------------------------------------------

    debug_patterns = [
        r"\bDEBUG\s*[:=]\s*true\b",
        r"\bdevelopment\b",
        r"\bstaging\b",
        r"\blocalhost\b",
        r"\b127\.0\.0\.1\b",
    ]

    for pattern in debug_patterns:

        if re.search(pattern, text, re.IGNORECASE):

            add_finding(
                title="Potential development/debug configuration exposed",
                severity="Low",
                category="Code / Configuration",
                description=(
                    "A development/debug-like value was observed in "
                    "public JavaScript."
                ),
                url=url,
                confidence="Low",
            )


# ============================================================
# 15. XSS REFLECTION TEST
# ============================================================

def xss_reflection_test(url):

    """
    Uses a harmless unique canary.

    This does NOT attempt to execute JavaScript.
    It determines whether user-controlled query input is reflected.
    """

    parsed = urlparse(url)
    params = parse_qs(
        parsed.query,
        keep_blank_values=True
    )

    if not params:
        return

    for parameter in params:

        canary = (
            "MILAEGIS_"
            + hashlib.sha256(
                f"{url}:{parameter}".encode()
            ).hexdigest()[:12]
        )

        test_params = dict(params)
        test_params[parameter] = [canary]

        query = urlencode(
            test_params,
            doseq=True
        )

        test_url = urlunparse((
            parsed.scheme,
            parsed.netloc,
            parsed.path,
            "",
            query,
            "",
        ))

        if not same_target_host(test_url):
            continue

        try:

            response = session.get(
                test_url,
                timeout=REQUEST_TIMEOUT
            )

        except Exception:
            continue

        if canary in response.text:

            # Reflection is not automatically XSS.
            add_finding(
                title="User input reflected in HTTP response",
                severity="Medium",
                category="XSS",
                description=(
                    "A unique canary supplied through a URL parameter "
                    "was reflected in the response. Reflection alone "
                    "does not prove executable XSS."
                ),
                url=test_url,
                confidence="Medium",
                evidence=parameter,
                cwe="CWE-79",
            )


# ============================================================
# 16. CSRF INDICATOR ANALYSIS
# ============================================================

def csrf_analysis():

    print("\n[+] CSRF indicator analysis")

    csrf_names = [
        "csrf",
        "xsrf",
        "csrf_token",
        "xsrf_token",
        "_token",
    ]

    for form in REPORT["forms"]:

        method = form["method"]

        if method not in ("POST", "PUT", "PATCH", "DELETE"):
            continue

        names = [
            (x.get("name") or "").lower()
            for x in form["inputs"]
        ]

        token_present = any(
            any(
                csrf_name in name
                for csrf_name in csrf_names
            )
            for name in names
        )

        if not token_present:

            add_finding(
                title="State-changing form without obvious CSRF token",
                severity="Low",
                category="CSRF",
                description=(
                    "No conventional CSRF token field was observed. "
                    "This is only an indicator; SameSite cookies, "
                    "Origin validation, or other mechanisms may provide "
                    "protection."
                ),
                url=form["action"],
                confidence="Low",
                cwe="CWE-352",
            )


# ============================================================
# 17. OPEN REDIRECT INDICATORS
# ============================================================

def open_redirect_analysis():

    redirect_parameters = {
        "redirect",
        "redirect_uri",
        "return",
        "return_url",
        "next",
        "continue",
        "url",
        "target",
    }

    for page in REPORT["pages"]:

        url = page["url"]

        params = parse_qs(
            urlparse(url).query
        )

        for parameter in params:

            if parameter.lower() in redirect_parameters:

                add_finding(
                    title="Potential open-redirect parameter",
                    severity="Low",
                    category="Open Redirect",
                    description=(
                        f"The URL contains a redirect-like parameter "
                        f"named '{parameter}'. Server-side validation "
                        "should be verified."
                    ),
                    url=url,
                    confidence="Low",
                    cwe="CWE-601",
                )


# ============================================================
# 18. TECHNOLOGY FINGERPRINTING
# ============================================================

def technology_fingerprint(response):

    if not response:
        return

    headers = {
        k.lower(): v
        for k, v in response.headers.items()
    }

    technologies = []

    server = headers.get("server")

    if server:
        technologies.append({
            "technology": server,
            "source": "HTTP Server header",
            "confidence": "Medium",
        })

    powered = headers.get("x-powered-by")

    if powered:
        technologies.append({
            "technology": powered,
            "source": "X-Powered-By",
            "confidence": "Medium",
        })

    for technology in technologies:

        REPORT["technologies"].append(
            technology
        )


# ============================================================
# 19. TLS INSPECTION
# ============================================================

def tls_inspection(host):

    print("\n[+] TLS inspection")

    context = ssl.create_default_context()

    try:

        with socket.create_connection(
            (host, 443),
            timeout=5
        ) as sock:

            with context.wrap_socket(
                sock,
                server_hostname=host
            ) as tls:

                certificate = tls.getpeercert()

                tls_info = {
                    "version": tls.version(),
                    "cipher": tls.cipher(),
                    "certificate_subject":
                        certificate.get("subject"),
                    "certificate_issuer":
                        certificate.get("issuer"),
                }

                REPORT["assets"].append({
                    "tls": tls_info
                })

                print(
                    "    TLS:",
                    tls.version()
                )

    except Exception as exc:

        print(
            "    TLS inspection failed:",
            exc
        )


# ============================================================
# 20. BOUNDED RESILIENCE TESTER
# ============================================================

@dataclass
class ProbeResult:
    status: int | None
    latency: float
    error: str | None = None


class BoundedResilienceTester:

    def __init__(self, target):

        self.target = target

    def request_once(self):

        started = time.monotonic()

        try:

            response = session.get(
                self.target,
                timeout=MAX_RESILIENCE_TIMEOUT,
                allow_redirects=True,
                headers={
                    "User-Agent":
                        USER_AGENT
                        + "-Resilience"
                }
            )

            return ProbeResult(
                status=response.status_code,
                latency=time.monotonic() - started
            )

        except Exception as exc:

            return ProbeResult(
                status=None,
                latency=time.monotonic() - started,
                error=str(exc)
            )

    def baseline(self, samples=5):

        results = []

        for _ in range(samples):

            results.append(
                self.request_once()
            )

            time.sleep(
                NORMAL_REQUEST_INTERVAL
            )

        latencies = [
            r.latency
            for r in results
            if r.status is not None
        ]

        if not latencies:

            raise RuntimeError(
                "Could not establish HTTP baseline."
            )

        return {
            "p50":
                statistics.median(latencies),

            "p95":
                max(latencies),

            "results":
                results,
        }

    def safety_triggered(
        self,
        results,
        baseline_latency
    ):

        if not results:
            return False, None

        # ----------------------------------------------------
        # Connection failure guard
        # ----------------------------------------------------

        if len(results) >= MAX_CONSECUTIVE_FAILURES:

            tail = results[
                -MAX_CONSECUTIVE_FAILURES:
            ]

            if all(
                r.status is None
                for r in tail
            ):

                return (
                    True,
                    "Consecutive connection failures"
                )

        # ----------------------------------------------------
        # HTTP 5xx guard
        # ----------------------------------------------------

        server_errors = [
            r for r in results
            if r.status is not None
            and 500 <= r.status <= 599
        ]

        if (
            len(server_errors) /
            len(results)
            >= MAX_5XX_RATIO
        ):

            return (
                True,
                "5xx error threshold exceeded"
            )

        # ----------------------------------------------------
        # Latency guard
        # ----------------------------------------------------

        valid = [
            r.latency
            for r in results
            if r.status is not None
        ]

        if valid and baseline_latency:

            current_latency = (
                statistics.median(valid)
            )

            if current_latency >= (
                baseline_latency *
                MAX_LATENCY_MULTIPLIER
            ):

                return (
                    True,
                    "Latency degradation threshold exceeded"
                )

        return False, None

    def run(self):

        print(
            "\n[+] Starting bounded resilience assessment"
        )

        print(
            f"    Maximum requests : "
            f"{MAX_RESILIENCE_REQUESTS}"
        )

        print(
            f"    Maximum duration : "
            f"{MAX_RESILIENCE_DURATION}s"
        )

        # ----------------------------------------------------
        # Baseline
        # ----------------------------------------------------

        baseline = self.baseline()

        baseline_latency = baseline["p50"]

        results = []

        started = time.monotonic()

        stop_reason = None

        # Deliberately small, fixed bursts.
        burst_sizes = [
            5,
            10,
            10,
            10,
            10,
        ]

        for burst_size in burst_sizes:

            # Global request ceiling.
            if (
                len(results)
                >= MAX_RESILIENCE_REQUESTS
            ):

                stop_reason = (
                    "Maximum request limit reached"
                )

                break

            # Global duration ceiling.
            if (
                time.monotonic() - started
                >= MAX_RESILIENCE_DURATION
            ):

                stop_reason = (
                    "Maximum duration reached"
                )

                break

            burst_size = min(
                burst_size,
                MAX_RESILIENCE_BURST,
                MAX_RESILIENCE_REQUESTS
                - len(results),
            )

            # ------------------------------------------------
            # Bounded concurrency
            # ------------------------------------------------

            with ThreadPoolExecutor(
                max_workers=MAX_RESILIENCE_CONCURRENCY
            ) as executor:

                futures = [
                    executor.submit(
                        self.request_once
                    )
                    for _ in range(burst_size)
                ]

                burst_results = [
                    future.result()
                    for future in as_completed(
                        futures
                    )
                ]

            results.extend(
                burst_results
            )

            should_stop, reason = (
                self.safety_triggered(
                    results,
                    baseline_latency
                )
            )

            if should_stop:

                stop_reason = reason

                print(
                    "    SAFETY STOP:",
                    reason
                )

                break

            # ------------------------------------------------
            # Cool-down between bursts
            # ------------------------------------------------

            time.sleep(
                NORMAL_REQUEST_INTERVAL
            )

        # ----------------------------------------------------
        # Mandatory recovery period
        # ----------------------------------------------------

        print(
            f"    Cooling down for "
            f"{RESILIENCE_COOLDOWN}s..."
        )

        time.sleep(
            RESILIENCE_COOLDOWN
        )

        # ----------------------------------------------------
        # Recovery measurement
        # ----------------------------------------------------

        recovery = self.baseline(
            samples=3
        )

        result = {
            "baseline_p50":
                baseline["p50"],

            "baseline_p95":
                baseline["p95"],

            "requests_sent":
                len(results),

            "results":
                [asdict(r) for r in results],

            "stop_reason":
                stop_reason,

            "recovery_p50":
                recovery["p50"],

            "recovery_p95":
                recovery["p95"],

            "duration":
                time.monotonic() - started,
        }

        # ----------------------------------------------------
        # Calculate observed behavior
        # ----------------------------------------------------

        valid = [
            r for r in results
            if r.status is not None
        ]

        status_codes = [
            r.status
            for r in valid
        ]

        result["status_distribution"] = (
            dict(
                pd.Series(
                    status_codes
                ).value_counts()
            )
            if status_codes
            else {}
        )

        result["rate_limiting_observed"] = (
            429 in status_codes
        )

        result["server_errors"] = sum(
            1
            for status in status_codes
            if 500 <= status <= 599
        )

        result["connection_failures"] = sum(
            1
            for r in results
            if r.status is None
        )

        REPORT["resilience"] = result

        return result


# ============================================================
# 21. CVE LOOKUP
# ============================================================

def lookup_cves(keyword):

    """
    Query NVD for a product/technology name.

    NVD availability/rate limits can change, so failures are
    treated as informational rather than fatal.
    """

    print(
        f"\n[+] CVE lookup: {keyword}"
    )

    try:

        response = requests.get(
            "https://services.nvd.nist.gov/rest/json/cves/2.0",
            params={
                "keywordSearch": keyword,
                "resultsPerPage": 5,
            },
            headers={
                "User-Agent": USER_AGENT
            },
            timeout=15,
        )

        if response.status_code != 200:

            print(
                "    NVD response:",
                response.status_code
            )

            return []

        data = response.json()

        results = []

        for vulnerability in data.get(
            "vulnerabilities",
            []
        ):

            cve = vulnerability.get(
                "cve",
                {}
            )

            metrics = cve.get(
                "metrics",
                {}
            )

            cvss = None

            # Prefer CVSS v3.1.
            if metrics.get("cvssMetricV31"):

                cvss = (
                    metrics["cvssMetricV31"][0]
                    .get("cvssData", {})
                    .get("baseScore")
                )

            elif metrics.get("cvssMetricV30"):

                cvss = (
                    metrics["cvssMetricV30"][0]
                    .get("cvssData", {})
                    .get("baseScore")
                )

            record = {
                "id": cve.get("id"),
                "published":
                    cve.get("published"),
                "lastModified":
                    cve.get("lastModified"),
                "cvss": cvss,
                "description":
                    (
                        cve.get(
                            "descriptions",
                            [{}]
                        )[0]
                        .get("value")
                    ),
            }

            results.append(record)

        REPORT["cves"].extend(results)

        return results

    except Exception as exc:

        print(
            "    NVD lookup failed:",
            exc
        )

        return []


# ============================================================
# 22. CISA KEV CORRELATION
# ============================================================

def load_kev_catalog():

    print(
        "\n[+] Loading CISA KEV catalog"
    )

    try:

        response = requests.get(
            "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json",
            timeout=20,
            headers={
                "User-Agent": USER_AGENT
            }
        )

        response.raise_for_status()

        data = response.json()

        kev_ids = {
            item["cveID"]
            for item in data.get(
                "vulnerabilities",
                []
            )
        }

        return kev_ids

    except Exception as exc:

        print(
            "    KEV lookup failed:",
            exc
        )

        return set()


def correlate_kev():

    kev_ids = load_kev_catalog()

    for cve in REPORT["cves"]:

        cve["kev"] = (
            cve["id"] in kev_ids
        )

        if cve["kev"]:

            REPORT["kev"].append(
                cve["id"]
            )


# ============================================================
# 23. RISK PRIORITIZATION
# ============================================================

def calculate_priority(finding):

    severity = finding.get(
        "severity",
        "Info"
    )

    confidence = finding.get(
        "confidence",
        "Medium"
    )

    score = severity_rank(
        severity
    ) * 20

    confidence_bonus = {
        "High": 20,
        "Medium": 10,
        "Low": 0,
    }.get(
        confidence,
        0
    )

    score += confidence_bonus

    # Internet-facing target gets an exposure bonus.
    score += 10

    if score >= 80:
        return "P1"

    if score >= 60:
        return "P2"

    if score >= 40:
        return "P3"

    return "P4"


def prioritize_findings():

    for finding in REPORT["findings"]:

        finding["priority"] = (
            calculate_priority(
                finding
            )
        )

    REPORT["findings"].sort(
        key=lambda x: (
            severity_rank(
                x["severity"]
            ),
            x["priority"],
        ),
        reverse=True
    )


# ============================================================
# 24. RUN ASSESSMENT
# ============================================================

def run_assessment():

    print("\n")
    print("=" * 70)
    print("MIL-AEGIS ASSESSMENT STARTING")
    print("=" * 70)

    addresses = dns_discovery()

    if addresses:

        host = addresses[0]

        open_ports = port_scan(
            host
        )

        for port in open_ports:

            service_fingerprint(
                host,
                port
            )

        tls_inspection(
            TARGET_HOST
        )

    crawl()

    # XSS reflection analysis only against URLs already
    # discovered on the authorized target.
    print(
        "\n[+] XSS reflection analysis"
    )

    tested = set()

    for page in REPORT["pages"]:

        url = page["url"]

        if (
            "?" in url
            and url not in tested
        ):

            tested.add(url)

            xss_reflection_test(
                url
            )

    csrf_analysis()

    open_redirect_analysis()

    # Fingerprint the initial response.
    try:

        response = session.get(
            TARGET,
            timeout=REQUEST_TIMEOUT
        )

        technology_fingerprint(
            response
        )

    except Exception:
        pass

    # --------------------------------------------------------
    # CVE correlation for explicitly observed technologies.
    #
    # Do not invent versions when a server doesn't expose one.
    # --------------------------------------------------------

    for technology in REPORT[
        "technologies"
    ]:

        name = technology[
            "technology"
        ]

        # Only query reasonably specific fingerprints.
        if name:

            lookup_cves(
                name
            )

    correlate_kev()

    # --------------------------------------------------------
    # Risk prioritization
    # --------------------------------------------------------

    prioritize_findings()

    print("\n[+] Running bounded resilience assessment")

    resilience = BoundedResilienceTester(
        TARGET
    ).run()

    print("\n")
    print("=" * 70)
    print("ASSESSMENT COMPLETE")
    print("=" * 70)

    print(
        "Pages:",
        len(REPORT["pages"])
    )

    print(
        "Ports:",
        len(REPORT["ports"])
    )

    print(
        "Forms:",
        len(REPORT["forms"])
    )

    print(
        "Cookies:",
        len(REPORT["cookies"])
    )

    print(
        "Findings:",
        len(REPORT["findings"])
    )

    print(
        "CVEs:",
        len(REPORT["cves"])
    )

    print(
        "KEV CVEs:",
        len(REPORT["kev"])
    )

    print(
        "Resilience requests:",
        resilience["requests_sent"]
    )

    return REPORT


# ============================================================
# 25. EXPORT JSON
# ============================================================

def save_report(filename="mil_aegis_report.json"):

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            REPORT,
            f,
            indent=2,
            default=str
        )

    print(
        f"\nReport saved to: {filename}"
    )


# ============================================================
# 26. SIMPLE RESULTS TABLE
# ============================================================

def show_findings():

    if not REPORT["findings"]:

        print(
            "\nNo findings generated."
        )

        return

    rows = []

    for finding in REPORT["findings"]:

        rows.append({
            "Priority":
                finding.get("priority"),

            "Severity":
                finding.get("severity"),

            "Category":
                finding.get("category"),

            "Finding":
                finding.get("title"),

            "Confidence":
                finding.get("confidence"),

            "URL":
                finding.get("url"),
        })

    df = pd.DataFrame(rows)

    display(df)


# ============================================================
# 27. EXECUTE
# ============================================================

REPORT = run_assessment()

save_report()

show_findings()

print("\nAssessment JSON is available as:")
print("mil_aegis_report.json")